In [76]:
!git clone https://github.com/KekeliIsHere/GreenPulse.git

Cloning into 'GreenPulse'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 22 (delta 2), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), done.
Resolving deltas: 100% (2/2), done.


In [77]:
%cd GreenPulse

/content/GreenPulse


In [78]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [55]:
import os
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F


from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cpu


In [56]:
import kagglehub

plantvillage_path = kagglehub.dataset_download(
    "abdallahalidev/plantvillage-dataset"
)

plant_disease_path = kagglehub.dataset_download(
    "karagwaanntreasure/plant-disease-detection"
)

print(plantvillage_path)
print(plant_disease_path)

Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
Using Colab cache for faster access to the 'plant-disease-detection' dataset.
/kaggle/input/plantvillage-dataset
/kaggle/input/plant-disease-detection


In [57]:
plantvillage_color = Path(
    plantvillage_path
) / "plantvillage dataset" / "color"


second_dataset = Path(
    plant_disease_path
) / "Dataset"


output_dataset = Path(
    "/content/crop_dataset"
)

output_dataset.mkdir(exist_ok=True)

In [58]:
classes_to_copy = [

    # PlantVillage names
    "Corn_(maize)___healthy",
    "Corn_(maize)___Common_rust",
    "Corn_(maize)___Northern_Leaf_Blight",

    "Tomato___healthy",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold"
]

In [59]:
def copy_matching_classes(source_folder, classes, destination):

    available = os.listdir(source_folder)

    for wanted in classes:

        matches = [
            folder for folder in available
            if wanted.lower().replace("_","")
            in folder.lower().replace("_","")
        ]

        if matches:

            folder = matches[0]

            source = source_folder / folder
            target = destination / folder


            shutil.copytree(
                source,
                target,
                dirs_exist_ok=True
            )

            print("Copied:", folder)

        else:
            print("Not found:", wanted)

In [60]:
copy_matching_classes(
    plantvillage_color,
    classes_to_copy,
    output_dataset
)

Copied: Corn_(maize)___healthy
Copied: Corn_(maize)___Common_rust_
Copied: Corn_(maize)___Northern_Leaf_Blight
Copied: Tomato___healthy
Copied: Tomato___Early_blight
Copied: Tomato___Late_blight
Copied: Tomato___Leaf_Mold


In [61]:
copy_matching_classes(
    second_dataset,
    classes_to_copy,
    output_dataset
)

Copied: Corn_(maize)___healthy
Copied: Corn_(maize)___Common_rust_
Copied: Corn_(maize)___Northern_Leaf_Blight
Copied: Tomato_healthy
Copied: Tomato_Early_blight
Copied: Tomato_Late_blight
Copied: Tomato_Leaf_Mold


In [62]:
for folder in output_dataset.iterdir():

    print(
        folder.name,
        "=>",
        len(list(folder.iterdir())),
        "images"
    )

Tomato_healthy => 1591 images
Tomato___Leaf_Mold => 952 images
Corn_(maize)___healthy => 2085 images
Tomato_Late_blight => 1909 images
Tomato___healthy => 1591 images
Tomato___Late_blight => 1909 images
Corn_(maize)___Northern_Leaf_Blight => 2133 images
Corn_(maize)___Common_rust_ => 2144 images
Tomato_Early_blight => 1000 images
Tomato___Early_blight => 1000 images
Tomato_Leaf_Mold => 952 images


In [63]:
transform = transforms.Compose([

    transforms.Resize(
        (224,224)
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])

In [64]:
dataset = ImageFolder(
    root=output_dataset,
    transform=transform
)


print("Classes:")
print(dataset.classes)

print(
    "Total images:",
    len(dataset)
)

Classes:
['Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___healthy', 'Tomato_healthy']
Total images: 17266


In [65]:
total_size = len(dataset)

train_size = int(0.7 * total_size)

val_size = int(0.15 * total_size)

test_size = (
    total_size
    - train_size
    - val_size
)


train_ds, val_ds, test_ds = random_split(

    dataset,

    [
        train_size,
        val_size,
        test_size
    ],

    generator=torch.Generator().manual_seed(42)
)


print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

12086
2589
2591


In [66]:
BATCH_SIZE = 32


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True
)


val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)


test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [67]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [68]:
from torchvision.models import mobilenet_v3_large
from torchvision.models import MobileNet_V3_Large_Weights

weights = MobileNet_V3_Large_Weights.DEFAULT

model = mobilenet_v3_large(weights=weights)

In [70]:
NUM_CLASSES = len(dataset.classes)

model.classifier[3] = nn.Linear(

    model.classifier[3].in_features,

    NUM_CLASSES

)

In [71]:
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [72]:
print("Trainable parameters:")
for name, param in model.named_parameters():
  if param.requires_grad:
    print(f" {name}")

Trainable parameters:
 classifier.0.weight
 classifier.0.bias
 classifier.3.weight
 classifier.3.bias


In [73]:

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()           # clear old gradients
        outputs = model(images)         # forward pass -> logits [batch, 10]
        loss = criterion(outputs, labels)
        loss.backward()                 # backprop
        optimizer.step()                # update weights

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()                        # no gradients needed for evaluation
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    avg_loss = running_loss / total
    accuracy = correct / total
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return avg_loss, accuracy, preds, labels

In [74]:
def fit(model, epochs=5, lr=1e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"Epoch {epoch:2d}/{epochs} | "
              f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f}")
    return history

In [ ]:
def plot_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(epochs, history["train_loss"], "o-", label="train")
    ax1.plot(epochs, history["val_loss"],   "o-", label="val")
    ax1.set_title("Loss"); ax1.set_xlabel("epoch"); ax1.legend()

    ax2.plot(epochs, history["train_acc"], "o-", label="train")
    ax2.plot(epochs, history["val_acc"],   "o-", label="val")
    ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()

    plt.tight_layout(); plt.show()

plot_curves(history)

In [75]:
torch.save(
    model.state_dict(),
    "crop_disease_model.pth"
)